# 🚀 Модуль 7. Урок 5. Принятие бизнес-решений по результатам A/B тестов и раскатка (Rollout)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Enlyl/data-analyst-2027/blob/main/course/module_07_ab_testing/lesson_05_business_decision_rollout.ipynb)

## 📌 Бизнес-контекст и методология
Статистическая значимость не равна экономической целесообразности.
При принятии решения о раскатке учитываются:
1. Затраты на поддержку новой фичи.
2. Эффект каннибализации других продуктов.
3. Риски деградации Guardrail метрик.

In [1]:
# 🚀 Универсальная инициализация окружения и данных
import os
import urllib.request
import sqlite3
import pandas as pd
import numpy as np

def ensure_dataset(filename, subfolder=""):
    """Автоматически находит датасет локально или скачивает с GitHub репозитория курса."""
    if os.path.exists(filename):
        return filename
    candidates = [
        os.path.join("course", "datasets", subfolder, filename),
        os.path.join("course", "datasets", filename),
        os.path.join("..", "datasets", subfolder, filename),
        os.path.join("..", "datasets", filename),
        os.path.join("..", "..", "course", "datasets", subfolder, filename),
        os.path.join("datasets", subfolder, filename),
        os.path.join("datasets", filename),
    ]
    for c in candidates:
        if os.path.exists(c):
            return c
    raw_base = "https://raw.githubusercontent.com/Enlyl/data-analyst-2027/main/course/datasets"
    url = f"{raw_base}/{subfolder}/{filename}" if subfolder else f"{raw_base}/{filename}"
    print(f"📥 Загрузка {filename} из репозитория курса...")
    urllib.request.urlretrieve(url, filename)
    print(f"✅ Датасет {filename} успешно готов!")
    return filename

# Загрузка ключевых датасетов
customers_csv = ensure_dataset("customers.csv", subfolder="raw")
orders_csv = ensure_dataset("orders.csv", subfolder="raw")
saas_csv = ensure_dataset("saas_users.csv", subfolder="raw")
print("🟢 Окружение готово к анализу данных.")


🟢 Окружение готово к анализу данных.


## 🔍 Разбор механики и демонстрация на реальных данных

In [2]:
# Оценка годового экономического эффекта от фичи
uplift_cr = 0.004           # +0.4% к конверсии
monthly_visitors = 500_000
avg_order_value = 3500.0    # 3500 руб.
margin_rate = 0.25          # 25% маржинальность

additional_orders_month = monthly_visitors * uplift_cr
additional_gross_profit = additional_orders_month * avg_order_value * margin_rate
annual_impact = additional_gross_profit * 12

print(f"Дополнительных заказов в месяц: {additional_orders_month:,.0f}")
print(f"Дополнительная маржинальная прибыль в год: {annual_impact:,.2f} руб.")


Дополнительных заказов в месяц: 2,000
Дополнительная маржинальная прибыль в год: 21,000,000.00 руб.


### 🎯 Практическое задание 1: Расчет ROI раскатки
Разработка и поддержка фичи стоят 5 000 000 руб. в год. Рассчитайте чистый годовой ROI.

In [3]:
### ВАШ КОД ЗДЕСЬ
cost_annual = 5_000_000.0
net_profit = annual_impact - cost_annual
roi = net_profit / cost_annual

print(f"Чистая прибыль: {net_profit:,.2f} руб. ROI: {roi:.1%}")
assert roi > 0, "Фича должна окупаться"
print("✅ Тест успешно пройден: экономическое обоснование готово!")


Чистая прибыль: 16,000,000.00 руб. ROI: 320.0%
✅ Тест успешно пройден: экономическое обоснование готово!


### 💡 Культура Rollout
Поэтапная раскатка (10% -> 25% -> 50% -> 100%) минимизирует риски инфраструктурных аварий.